# CatBoost Modelling for HDB Resale Price Prediction

This notebook compares baseline and Optuna-tuned CatBoost models under two feature sets:

1. **Standard feature set**
2. **Standard + `dist_cbd_m` feature set**

For each setup, the notebook evaluates performance on:
- the **outer validation set** for model comparison
- the **2026 out-of-sample test set** for final holdout assessment

In [28]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from catboost import CatBoostRegressor
import optuna

## 1. Load and clean pre-2026 data

This section loads the pre-2026 dataset used for model development and applies the same basic cleaning steps throughout the notebook:

1. Drop rows with missing values
2. Drop duplicate rows
3. Create the log-transformed target variable `log_resale_price_real`

In [29]:
df_pre2026_raw = pd.read_csv("../../merged_data/[FINAL]hdb_with_amenities_macro_pre2026.csv")
print(f"Initial shape: {df_pre2026_raw.shape}")

df_pre2026 = df_pre2026_raw.dropna()
print(f"After dropping nulls: {df_pre2026.shape} (dropped {len(df_pre2026_raw) - len(df_pre2026)})")

# Remove duplicate rows to avoid repeated transactions affecting model training
n_before = len(df_pre2026)
df_pre2026 = df_pre2026.drop_duplicates()
print(f"After dropping duplicates: {df_pre2026.shape} (dropped {n_before - len(df_pre2026)})")

# Log-transform target while preserving the original price column for evaluation in dollar space
df_pre2026["log_resale_price_real"] = np.log(df_pre2026["resale_price_real"])

Initial shape: (134479, 37)
After dropping nulls: (134301, 37) (dropped 178)
After dropping duplicates: (134211, 37) (dropped 90)


## 2. Create the outer development / validation split

This is the **outer split** used for fair model comparison.

- `dev_train_df`: used for model development
- `model_val_df`: held-out validation set used only for comparing model variants

The split is stratified by `town + flat_type + year` to preserve the joint distribution of key market segments.

In [30]:
# =========================================================
# OUTER DEVELOPMENT / VALIDATION SPLIT
# =========================================================
# dev_train_df: used for model development
# model_val_df: held-out validation set used only for model comparison
# =========================================================

df_pre2026["strat_key"] = (
    df_pre2026["town"].astype(str) + "_" +
    df_pre2026["flat_type"].astype(str) + "_" +
    df_pre2026["year"].astype(str)
)

# Remove singleton strata because stratified split requires at least 2 rows per stratum
strat_counts = df_pre2026["strat_key"].value_counts()
valid_strat_keys = strat_counts[strat_counts >= 2].index

n_before = len(df_pre2026)
df_pre2026 = df_pre2026[df_pre2026["strat_key"].isin(valid_strat_keys)].copy()
print(f"Dropped {n_before - len(df_pre2026)} rows with singleton strat_key combinations. Remaining: {len(df_pre2026):,}")

dev_train_df, model_val_df = train_test_split(
    df_pre2026,
    test_size=0.2,
    random_state=42,
    stratify=df_pre2026["strat_key"]
)

print(f"Development train size: {len(dev_train_df):,}")
print(f"Model validation size: {len(model_val_df):,}")

print("\nYear distribution (%):")
dev_year_dist = dev_train_df["year"].value_counts(normalize=True).sort_index().rename("Dev Train %")
model_val_year_dist = model_val_df["year"].value_counts(normalize=True).sort_index().rename("Model Val %")
year_distribution = pd.concat([dev_year_dist, model_val_year_dist], axis=1)
print(year_distribution.map(lambda x: f"{x:.2%}"))

print("\nFlat type distribution (%):")
dev_flat_type_dist = dev_train_df["flat_type"].value_counts(normalize=True).rename("Dev Train %")
model_val_flat_type_dist = model_val_df["flat_type"].value_counts(normalize=True).rename("Model Val %")
flat_type_distribution = pd.concat([dev_flat_type_dist, model_val_flat_type_dist], axis=1)
print(flat_type_distribution.map(lambda x: f"{x:.2%}"))

Dropped 6 rows with singleton strat_key combinations. Remaining: 134,205
Development train size: 107,364
Model validation size: 26,841

Year distribution (%):
     Dev Train % Model Val %
year                        
2021      21.61%      21.61%
2022      19.86%      19.87%
2023      19.15%      19.16%
2024      20.71%      20.70%
2025      18.67%      18.67%

Flat type distribution (%):
                 Dev Train % Model Val %
flat_type                               
4 ROOM                43.02%      43.00%
5 ROOM                24.25%      24.23%
3 ROOM                23.71%      23.71%
EXECUTIVE              6.62%       6.63%
2 ROOM                 2.35%       2.36%
1 ROOM                 0.03%       0.03%
MULTI-GENERATION       0.03%       0.03%


## 3. Define the standard feature set

This section prepares the **standard feature set** used in the baseline CatBoost experiments.

- Continuous predictors: lease and amenity-distance variables
- Categorical predictors: `flat_type`, `town`, `floor_category`
- Excluded here: `dist_cbd_m`

In [ ]:
# =========================================================
# DEFINE STANDARD FEATURE SET
# =========================================================
standard_feature_cols = [
    "remaining_lease_years",
    "nearest_train_dist_m",
    "dist_nearest_hawker_m",
    "dist_nearest_primary_m",
    "num_primary_1km",
    "dist_nearest_park_m",
    "num_parks_1km",
    "dist_nearest_sportsg_m",
    "dist_nearest_mall_m",
    "dist_nearest_healthcare_m",
    "flat_type",
    "town",
    "floor_category",
]

categorical_feature_cols = ["flat_type", "town", "floor_category"]
target_col = "log_resale_price_real"

# Outer validation set for model comparison
X_model_val_standard = model_val_df[standard_feature_cols].copy()
y_model_val_log = model_val_df[target_col].copy()
y_model_val_actual = model_val_df["resale_price_real"].copy()

print("X_model_val_standard shape:", X_model_val_standard.shape)
print("Number of standard features:", len(standard_feature_cols))
print("Categorical features:", categorical_feature_cols)

X_model_val shape: (26841, 13)
Number of features: 13
Categorical features: ['flat_type', 'town', 'floor_category']


In [32]:
# =========================================================
# INNER TRAIN / EARLY-STOPPING SPLIT
# =========================================================
# fit_train_df: used to fit the model
# early_stop_val_df: used only for early stopping / hyperparameter tuning
# =========================================================

fit_train_df, early_stop_val_df = train_test_split(
    dev_train_df,
    test_size=0.2,
    random_state=42,
    stratify=dev_train_df["flat_type"]
)

print(f"Fit-train size: {len(fit_train_df):,}")
print(f"Early-stop validation size: {len(early_stop_val_df):,}")

X_fit_train_standard = fit_train_df[standard_feature_cols].copy()
y_fit_train_log = fit_train_df[target_col].copy()

X_early_stop_standard = early_stop_val_df[standard_feature_cols].copy()
y_early_stop_log = early_stop_val_df[target_col].copy()

standard_cat_feature_indices = [
    X_fit_train_standard.columns.get_loc(col) for col in categorical_feature_cols
]

print("X_fit_train_standard shape:", X_fit_train_standard.shape)
print("X_early_stop_standard shape:", X_early_stop_standard.shape)
print("Categorical feature indices:", standard_cat_feature_indices)

Fit train size: 85,891
Early stopping validation size: 21,473
X_fit_train shape: (85891, 15)
X_early_stop shape: (21473, 15)
Categorical feature indices: [10, 11, 12]


## 4. Standard feature set — baseline CatBoost on validation set

In [13]:
standard_basic_catboost_model = CatBoostRegressor(
    loss_function="RMSE",
    verbose=200
)

standard_basic_catboost_model.fit(
    X_fit_train_standard,
    y_fit_train_log,
    cat_features=standard_cat_feature_indices
)

Learning rate set to 0.082745
0:	learn: 0.2850442	total: 52.4ms	remaining: 52.3s
200:	learn: 0.0723878	total: 11.8s	remaining: 46.9s
400:	learn: 0.0639907	total: 23.9s	remaining: 35.6s
600:	learn: 0.0605333	total: 36s	remaining: 23.9s
800:	learn: 0.0582454	total: 49.1s	remaining: 12.2s
999:	learn: 0.0566423	total: 1m 2s	remaining: 0us


CatBoostRegressor(loss_function='RMSE', verbose=200)

In [ ]:
# =========================================================
# VALIDATION PREDICTIONS — STANDARD FEATURE SET / BASELINE MODEL
# =========================================================

y_model_val_pred_log = standard_basic_catboost_model.predict(X_model_val_standard)
y_model_val_pred = np.exp(y_model_val_pred_log)

# Optional training-set predictions for diagnostics
y_fit_train_pred_log = standard_basic_catboost_model.predict(X_fit_train_standard)
y_fit_train_pred = np.exp(y_fit_train_pred_log)
y_fit_train_actual = fit_train_df["resale_price_real"].copy()

In [ ]:
# =========================================================
# EVALUATION METRICS
# =========================================================
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

def linlin_loss(y_true, y_pred, underpredict_weight=2.0):
    """Asymmetric linear loss that penalises underprediction more than overprediction."""
    errors = np.array(y_true) - np.array(y_pred)  # positive = underprediction
    loss = np.where(errors > 0, underpredict_weight * errors, -errors)
    return np.mean(loss)

standard_basic_val_rmse = rmse(y_model_val_actual, y_model_val_pred)
standard_basic_val_linlin = linlin_loss(y_model_val_actual, y_model_val_pred, underpredict_weight=2.0)
standard_basic_val_r2 = r2_score(y_model_val_actual, y_model_val_pred)

print("=== STANDARD FEATURE SET | BASELINE CATBOOST | OUTER VALIDATION ===")
print(f"RMSE:         ${standard_basic_val_rmse:,.0f}")
print(f"R²:           {standard_basic_val_r2:.3f}")
print(f"Linlin Loss:  ${standard_basic_val_linlin:,.0f} (underpredict weight = 2.0)")

=== CATBOOST MODEL EVALUATION (OUTER VALIDATION SET) ===
RMSE:         $43,185
R²:           0.957
Linlin Loss:  $45,675 (underpredict weight = 2.0)


## 5. Standard feature set — Optuna-tuned CatBoost on validation set

In [ ]:
# =========================================================
# OPTUNA TUNING — STANDARD FEATURE SET
# =========================================================

def objective(trial):
    params = {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "iterations": trial.suggest_int("iterations", 500, 3000, step=250),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 30),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
        "random_seed": 42,
        "early_stopping_rounds": 100,
        "verbose": 0,
    }

    trial_model = CatBoostRegressor(**params)
    trial_model.fit(
        X_fit_train_standard,
        y_fit_train_log,
        cat_features=standard_cat_feature_indices,
        eval_set=(X_early_stop_standard, y_early_stop_log),
        use_best_model=True,
    )

    y_early_stop_pred_log = trial_model.predict(X_early_stop_standard)
    y_early_stop_pred = np.exp(y_early_stop_pred_log)
    y_early_stop_actual = np.exp(y_early_stop_log)

    return np.sqrt(mean_squared_error(y_early_stop_actual, y_early_stop_pred))

standard_study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
standard_study.optimize(objective, n_trials=30, show_progress_bar=True)

[I 2026-04-03 17:04:05,124] A new study created in memory with name: no-name-a8b6f354-2579-4794-b752-7a6c1c84a091


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-04-03 17:09:52,904] Trial 0 finished with value: 38996.27657449339 and parameters: {'iterations': 1500, 'learning_rate': 0.08927180304353628, 'depth': 9, 'l2_leaf_reg': 3.968793330444372, 'min_data_in_leaf': 5, 'subsample': 0.662397808134481, 'random_strength': 0.0017073967431528124, 'bagging_temperature': 4.330880728874676}. Best is trial 0 with value: 38996.27657449339.
[I 2026-04-03 17:12:53,677] Trial 1 finished with value: 47820.8293223652 and parameters: {'iterations': 2000, 'learning_rate': 0.051059032093947576, 'depth': 4, 'l2_leaf_reg': 9.330606024425668, 'min_data_in_leaf': 25, 'subsample': 0.6849356442713105, 'random_strength': 0.005337032762603957, 'bagging_temperature': 0.9170225492671691}. Best is trial 0 with value: 38996.27657449339.
[I 2026-04-03 17:16:13,932] Trial 2 finished with value: 43606.11767498852 and parameters: {'iterations': 1250, 'learning_rate': 0.03347776308515933, 'depth': 7, 'l2_leaf_reg': 1.9553708662745248, 'min_data_in_leaf': 19, 'subsample'

In [11]:
print("Best RMSE:", standard_study.best_value)

print("\nBest hyperparameters:")
for key, value in standard_study.best_params.items():
    print(f"{key}: {value}")

standard_best_params = standard_study.best_params.copy()
standard_best_params.update({
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "random_seed": 42,
    "verbose": 200,
    "early_stopping_rounds": 100,
})

standard_tuned_catboost_model = CatBoostRegressor(**standard_best_params)
standard_tuned_catboost_model.fit(
    X_fit_train_standard,
    y_fit_train_log,
    cat_features=standard_cat_feature_indices,
    eval_set=(X_early_stop_standard, y_early_stop_log),
    use_best_model=True,
)

Best RMSE: 38546.53041430045

Best hyperparameters:
iterations: 3000
learning_rate: 0.09977171260113843
depth: 9
l2_leaf_reg: 3.5647267085676377
min_data_in_leaf: 6
subsample: 0.7457769212953824
random_strength: 9.525499182950078
bagging_temperature: 3.656471675173943


### 5.1 Generate validation predictions for the tuned standard-feature model

In [ ]:
# =========================================================
# VALIDATION PREDICTIONS — STANDARD FEATURE SET / OPTUNA-TUNED MODEL
# =========================================================

y_model_val_pred_log = standard_tuned_catboost_model.predict(X_model_val_standard)
y_model_val_pred = np.exp(y_model_val_pred_log)

y_fit_train_pred_log = standard_tuned_catboost_model.predict(X_fit_train_standard)
y_fit_train_pred = np.exp(y_fit_train_pred_log)
y_fit_train_actual = fit_train_df["resale_price_real"].copy()

### 5.2 Evaluate the tuned standard-feature model on the outer validation set

In [ ]:
standard_tuned_val_rmse = rmse(y_model_val_actual, y_model_val_pred)
standard_tuned_val_linlin = linlin_loss(y_model_val_actual, y_model_val_pred, underpredict_weight=2.0)
standard_tuned_val_r2 = r2_score(y_model_val_actual, y_model_val_pred)

print("=== STANDARD FEATURE SET | OPTUNA-TUNED CATBOOST | OUTER VALIDATION ===")
print(f"RMSE:         ${standard_tuned_val_rmse:,.0f}")
print(f"R²:           {standard_tuned_val_r2:.3f}")
print(f"Linlin Loss:  ${standard_tuned_val_linlin:,.0f} (underpredict weight = 2.0)")

=== CATBOOST MODEL EVALUATION (OUTER VALIDATION SET) ===
RMSE:         $38,942
R²:           0.965
Linlin Loss:  $40,651 (underpredict weight = 2.0)


## 6. Standard feature set — tuned CatBoost on 2026 out-of-sample test set

In [45]:
df_2026_raw = pd.read_csv("../../merged_data/[FINAL]hdb_with_amenities_macro_2026.csv")
print(f"Initial shape: {df_2026_raw.shape}")

df_2026 = df_2026_raw.dropna()
print(f"After dropping nulls: {df_2026.shape} (dropped {len(df_2026_raw) - len(df_2026)})")

n_before = len(df_2026)
df_2026 = df_2026.drop_duplicates()
print(f"After dropping duplicates: {df_2026.shape} (dropped {n_before - len(df_2026)})")

df_2026["log_resale_price_real"] = np.log(df_2026["resale_price_real"])

Initial shape: (6058, 37)
After dropping nulls: (6055, 37) (dropped 3)
After dropping duplicates: (6051, 37) (dropped 4)


In [25]:
X_test_standard = df_2026[standard_feature_cols].copy()
y_test_log = df_2026[target_col]
y_test_actual = df_2026["resale_price_real"]

y_test_pred_log = standard_tuned_catboost_model.predict(X_test_standard)
y_test_pred = np.exp(y_test_pred_log)

standard_tuned_test_rmse = rmse(y_test_actual, y_test_pred)
standard_tuned_test_linlin = linlin_loss(y_test_actual, y_test_pred, underpredict_weight=2.0)
standard_tuned_test_r2 = r2_score(y_test_actual, y_test_pred)

print("=== STANDARD FEATURE SET | OPTUNA-TUNED CATBOOST | 2026 OOS TEST ===")
print(f"R²:           {standard_tuned_test_r2:.3f}")
print(f"RMSE:         ${standard_tuned_test_rmse:,.0f}")
print(f"Linlin Loss:  ${standard_tuned_test_linlin:,.0f}")

Best RMSE: 38546.53041430045
Best parameters: {'iterations': 3000, 'learning_rate': 0.09977171260113843, 'depth': 9, 'l2_leaf_reg': 3.5647267085676377, 'min_data_in_leaf': 6, 'subsample': 0.7457769212953824, 'random_strength': 9.525499182950078, 'bagging_temperature': 3.656471675173943}
0:	learn: 0.2838471	test: 0.2852530	best: 0.2852530 (0)	total: 159ms	remaining: 7m 58s
200:	learn: 0.0617427	test: 0.0632648	best: 0.0632648 (200)	total: 26.6s	remaining: 6m 10s
400:	learn: 0.0543755	test: 0.0577546	best: 0.0577546 (400)	total: 53.8s	remaining: 5m 48s
600:	learn: 0.0512424	test: 0.0560943	best: 0.0560943 (600)	total: 1m 21s	remaining: 5m 24s
800:	learn: 0.0492313	test: 0.0553044	best: 0.0553044 (800)	total: 1m 48s	remaining: 4m 57s
1000:	learn: 0.0476879	test: 0.0548046	best: 0.0548046 (1000)	total: 2m 15s	remaining: 4m 31s
1200:	learn: 0.0464188	test: 0.0545009	best: 0.0545009 (1200)	total: 2m 43s	remaining: 4m 4s
1400:	learn: 0.0453027	test: 0.0543263	best: 0.0543263 (1400)	total: 3m 

CatBoostRegressor(bagging_temperature=3.656471675173943, depth=9, early_stopping_rounds=100, eval_metric='RMSE', iterations=3000, l2_leaf_reg=3.5647267085676377, learning_rate=0.09977171260113843, loss_function='RMSE', min_data_in_leaf=6, random_seed=42, random_strength=9.525499182950078, subsample=0.7457769212953824, verbose=200)

In [ ]:
standard_results = pd.DataFrame({
    "model_variant": ["baseline", "optuna_tuned", "optuna_tuned_oos"],
    "feature_set": ["standard", "standard", "standard"],
    "split": ["validation", "validation", "2026_test"],
    "rmse": [standard_basic_val_rmse, standard_tuned_val_rmse, standard_tuned_test_rmse],
    "r2": [standard_basic_val_r2, standard_tuned_val_r2, standard_tuned_test_r2],
    "linlin_loss": [standard_basic_val_linlin, standard_tuned_val_linlin, standard_tuned_test_linlin],
})
standard_results

=== 2026 TEST SET (OUT-OF-SAMPLE) ===
R^2: 0.965
RMSE: $39,602
Linlin Loss: $41,739


## 7. Extend the feature set by adding `dist_cbd_m`

### 7.1 Define the enhanced feature set

In [36]:
# =========================================================
# DEFINE ENHANCED FEATURE SET
# =========================================================
enhanced_feature_cols = [
    "remaining_lease_years",
    "nearest_train_dist_m",
    "dist_nearest_hawker_m",
    "dist_nearest_primary_m",
    "num_primary_1km",
    "dist_nearest_park_m",
    "num_parks_1km",
    "dist_nearest_sportsg_m",
    "dist_nearest_mall_m",
    "dist_nearest_healthcare_m",
    "dist_cbd_m",
    "flat_type",
    "town",
    "floor_category",
]

# Outer validation set for model comparison
X_model_val_enhanced = model_val_df[enhanced_feature_cols].copy()
y_model_val_log = model_val_df[target_col].copy()
y_model_val_actual = model_val_df["resale_price_real"].copy()

print("X_model_val_enhanced shape:", X_model_val_enhanced.shape)
print("Number of enhanced features:", len(enhanced_feature_cols))
print("Categorical features:", categorical_feature_cols)

X_model_val shape: (26841, 14)
Number of features: 14
Categorical features: ['flat_type', 'town', 'floor_category']


In [37]:
# =========================================================
# INNER TRAIN / EARLY-STOPPING SPLIT FOR ENHANCED FEATURE SET
# =========================================================

X_fit_train_enhanced = fit_train_df[enhanced_feature_cols].copy()
X_early_stop_enhanced = early_stop_val_df[enhanced_feature_cols].copy()

enhanced_cat_feature_indices = [
    X_fit_train_enhanced.columns.get_loc(col) for col in categorical_feature_cols
]

print("X_fit_train_enhanced shape:", X_fit_train_enhanced.shape)
print("X_early_stop_enhanced shape:", X_early_stop_enhanced.shape)
print("Categorical feature indices:", enhanced_cat_feature_indices)

Fit train size: 85,891
Early stopping validation size: 21,473
X_fit_train shape: (85891, 14)
X_early_stop shape: (21473, 14)
Categorical feature indices: [10, 11, 12]


## 8. Enhanced feature set — baseline CatBoost on validation set

In [38]:
enhanced_basic_catboost_model = CatBoostRegressor(
    loss_function="RMSE",
    verbose=200,
)

enhanced_basic_catboost_model.fit(
    X_fit_train_enhanced,
    y_fit_train_log,
    cat_features=enhanced_cat_feature_indices,
)

Learning rate set to 0.082745
0:	learn: 0.2832002	total: 103ms	remaining: 1m 42s
200:	learn: 0.0692786	total: 35.2s	remaining: 2m 20s
400:	learn: 0.0620842	total: 1m 1s	remaining: 1m 31s
600:	learn: 0.0587986	total: 1m 27s	remaining: 58.2s
800:	learn: 0.0567993	total: 1m 53s	remaining: 28.2s
999:	learn: 0.0554184	total: 2m 22s	remaining: 0us


CatBoostRegressor(loss_function='RMSE', verbose=200)

In [39]:
# =========================================================
# VALIDATION PREDICTIONS — ENHANCED FEATURE SET / BASELINE MODEL
# =========================================================

y_model_val_pred_log = enhanced_basic_catboost_model.predict(X_model_val_enhanced)
y_model_val_pred = np.exp(y_model_val_pred_log)

y_fit_train_pred_log = enhanced_basic_catboost_model.predict(X_fit_train_enhanced)
y_fit_train_pred = np.exp(y_fit_train_pred_log)
y_fit_train_actual = fit_train_df["resale_price_real"].copy()

In [40]:
enhanced_basic_val_rmse = rmse(y_model_val_actual, y_model_val_pred)
enhanced_basic_val_linlin = linlin_loss(y_model_val_actual, y_model_val_pred, underpredict_weight=2.0)
enhanced_basic_val_r2 = r2_score(y_model_val_actual, y_model_val_pred)

print("=== ENHANCED FEATURE SET | BASELINE CATBOOST | OUTER VALIDATION ===")
print(f"RMSE:         ${enhanced_basic_val_rmse:,.0f}")
print(f"R²:           {enhanced_basic_val_r2:.3f}")
print(f"Linlin Loss:  ${enhanced_basic_val_linlin:,.0f} (underpredict weight = 2.0)")

=== CATBOOST MODEL EVALUATION (OUTER VALIDATION SET) ===
RMSE:         $42,219
R²:           0.959
Linlin Loss:  $44,524 (underpredict weight = 2.0)


## 9. Enhanced feature set — Optuna-tuned CatBoost on validation set

In [41]:
# =========================================================
# OPTUNA TUNING — ENHANCED FEATURE SET
# =========================================================

def objective_enhanced(trial):
    params = {
        "loss_function": "RMSE",
        "eval_metric": "RMSE",
        "iterations": trial.suggest_int("iterations", 500, 3000, step=250),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 30),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
        "random_seed": 42,
        "early_stopping_rounds": 100,
        "verbose": 0,
    }

    trial_model = CatBoostRegressor(**params)
    trial_model.fit(
        X_fit_train_enhanced,
        y_fit_train_log,
        cat_features=enhanced_cat_feature_indices,
        eval_set=(X_early_stop_enhanced, y_early_stop_log),
        use_best_model=True,
    )

    y_early_stop_pred_log = trial_model.predict(X_early_stop_enhanced)
    y_early_stop_pred = np.exp(y_early_stop_pred_log)
    y_early_stop_actual = np.exp(y_early_stop_log)

    return np.sqrt(mean_squared_error(y_early_stop_actual, y_early_stop_pred))

enhanced_study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
enhanced_study.optimize(objective_enhanced, n_trials=30, show_progress_bar=True)

[I 2026-04-04 11:16:00,491] A new study created in memory with name: no-name-8e357af9-99eb-4b0e-8686-93edf91ee73e


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-04-04 11:21:52,248] Trial 0 finished with value: 38732.52256307493 and parameters: {'iterations': 1500, 'learning_rate': 0.08927180304353628, 'depth': 9, 'l2_leaf_reg': 3.968793330444372, 'min_data_in_leaf': 5, 'subsample': 0.662397808134481, 'random_strength': 0.0017073967431528124, 'bagging_temperature': 4.330880728874676}. Best is trial 0 with value: 38732.52256307493.
[I 2026-04-04 11:25:13,604] Trial 1 finished with value: 46114.66662074218 and parameters: {'iterations': 2000, 'learning_rate': 0.051059032093947576, 'depth': 4, 'l2_leaf_reg': 9.330606024425668, 'min_data_in_leaf': 25, 'subsample': 0.6849356442713105, 'random_strength': 0.005337032762603957, 'bagging_temperature': 0.9170225492671691}. Best is trial 0 with value: 38732.52256307493.
[I 2026-04-04 11:28:57,122] Trial 2 finished with value: 42844.41767311457 and parameters: {'iterations': 1250, 'learning_rate': 0.03347776308515933, 'depth': 7, 'l2_leaf_reg': 1.9553708662745248, 'min_data_in_leaf': 19, 'subsample

In [42]:
print("Best RMSE:", enhanced_study.best_value)

print("\nBest hyperparameters:")
for key, value in enhanced_study.best_params.items():
    print(f"{key}: {value}")

enhanced_best_params = enhanced_study.best_params.copy()
enhanced_best_params.update({
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "random_seed": 42,
    "verbose": 200,
    "early_stopping_rounds": 100,
})

enhanced_tuned_catboost_model = CatBoostRegressor(**enhanced_best_params)
enhanced_tuned_catboost_model.fit(
    X_fit_train_enhanced,
    y_fit_train_log,
    cat_features=enhanced_cat_feature_indices,
    eval_set=(X_early_stop_enhanced, y_early_stop_log),
    use_best_model=True,
)

Best RMSE: 38383.27580527543

Best hyperparameters:
iterations: 2750
learning_rate: 0.06290775092843544
depth: 8
l2_leaf_reg: 1.744847497692658
min_data_in_leaf: 3
subsample: 0.8437252794910377
random_strength: 0.457377568379274
bagging_temperature: 4.513943156645768


In [43]:
# =========================================================
# VALIDATION PREDICTIONS — ENHANCED FEATURE SET / OPTUNA-TUNED MODEL
# =========================================================

y_model_val_pred_log = enhanced_tuned_catboost_model.predict(X_model_val_enhanced)
y_model_val_pred = np.exp(y_model_val_pred_log)

y_fit_train_pred_log = enhanced_tuned_catboost_model.predict(X_fit_train_enhanced)
y_fit_train_pred = np.exp(y_fit_train_pred_log)
y_fit_train_actual = fit_train_df["resale_price_real"].copy()

In [44]:
enhanced_tuned_val_rmse = rmse(y_model_val_actual, y_model_val_pred)
enhanced_tuned_val_linlin = linlin_loss(y_model_val_actual, y_model_val_pred, underpredict_weight=2.0)
enhanced_tuned_val_r2 = r2_score(y_model_val_actual, y_model_val_pred)

print("=== ENHANCED FEATURE SET | OPTUNA-TUNED CATBOOST | OUTER VALIDATION ===")
print(f"RMSE:         ${enhanced_tuned_val_rmse:,.0f}")
print(f"R²:           {enhanced_tuned_val_r2:.3f}")
print(f"Linlin Loss:  ${enhanced_tuned_val_linlin:,.0f} (underpredict weight = 2.0)")

=== CATBOOST MODEL EVALUATION (OUTER VALIDATION SET) ===
RMSE:         $38,882
R²:           0.965
Linlin Loss:  $40,526 (underpredict weight = 2.0)


## 10. Enhanced feature set — tuned CatBoost on 2026 out-of-sample test set

In [49]:
X_test_enhanced = df_2026[enhanced_feature_cols].copy()
y_test_log = df_2026[target_col]
y_test_actual = df_2026["resale_price_real"]

y_test_pred_log = enhanced_tuned_catboost_model.predict(X_test_enhanced)
y_test_pred = np.exp(y_test_pred_log)

enhanced_tuned_test_rmse = rmse(y_test_actual, y_test_pred)
enhanced_tuned_test_linlin = linlin_loss(y_test_actual, y_test_pred, underpredict_weight=2.0)
enhanced_tuned_test_r2 = r2_score(y_test_actual, y_test_pred)

print("=== ENHANCED FEATURE SET | OPTUNA-TUNED CATBOOST | 2026 OOS TEST ===")
print(f"R²:           {enhanced_tuned_test_r2:.3f}")
print(f"RMSE:         ${enhanced_tuned_test_rmse:,.0f}")
print(f"Linlin Loss:  ${enhanced_tuned_test_linlin:,.0f}")

[11, 12, 13]


In [50]:
enhanced_results = pd.DataFrame({
    "model_variant": ["baseline", "optuna_tuned", "optuna_tuned_oos"],
    "feature_set": ["enhanced", "enhanced", "enhanced"],
    "split": ["validation", "validation", "2026_test"],
    "rmse": [enhanced_basic_val_rmse, enhanced_tuned_val_rmse, enhanced_tuned_test_rmse],
    "r2": [enhanced_basic_val_r2, enhanced_tuned_val_r2, enhanced_tuned_test_r2],
    "linlin_loss": [enhanced_basic_val_linlin, enhanced_tuned_val_linlin, enhanced_tuned_test_linlin],
})
enhanced_results

Best RMSE: 38383.27580527543
Best parameters: {'iterations': 2750, 'learning_rate': 0.06290775092843544, 'depth': 8, 'l2_leaf_reg': 1.744847497692658, 'min_data_in_leaf': 3, 'subsample': 0.8437252794910377, 'random_strength': 0.457377568379274, 'bagging_temperature': 4.513943156645768}
0:	learn: 0.2872572	test: 0.2887464	best: 0.2887464 (0)	total: 79.9ms	remaining: 3m 39s
200:	learn: 0.0646741	test: 0.0654858	best: 0.0654858 (200)	total: 19.3s	remaining: 4m 4s
400:	learn: 0.0580811	test: 0.0599182	best: 0.0599182 (400)	total: 55.7s	remaining: 5m 26s
600:	learn: 0.0549717	test: 0.0576455	best: 0.0576455 (600)	total: 1m 31s	remaining: 5m 27s
800:	learn: 0.0530155	test: 0.0564734	best: 0.0564734 (800)	total: 2m 5s	remaining: 5m 5s
1000:	learn: 0.0515536	test: 0.0557370	best: 0.0557370 (1000)	total: 2m 38s	remaining: 4m 37s
1200:	learn: 0.0504598	test: 0.0552336	best: 0.0552336 (1200)	total: 3m 14s	remaining: 4m 11s
1400:	learn: 0.0495276	test: 0.0548726	best: 0.0548726 (1400)	total: 3m 52

CatBoostRegressor(bagging_temperature=4.513943156645768, depth=8, early_stopping_rounds=100, eval_metric='RMSE', iterations=2750, l2_leaf_reg=1.744847497692658, learning_rate=0.06290775092843544, loss_function='RMSE', min_data_in_leaf=3, random_seed=42, random_strength=0.457377568379274, subsample=0.8437252794910377, verbose=200)

In [51]:
comparison_table = pd.concat([standard_results, enhanced_results], ignore_index=True)
comparison_table

=== 2026 TEST SET (OUT-OF-SAMPLE) ===
R^2: 0.965
RMSE: $39,210
Linlin Loss: $41,334
